# Fair Small-Data Ariel Comparison

This notebook compares the actual `codex` Ariel quantum regressor against the actual `sota` FMPE model on the same small-data split.

Fairness rules:
- Both models use the exact same labeled `train`, `validation`, and `holdout` rows.
- Each branch fits its own scalers using only the shared training rows.
- Final comparison uses holdout RMSE, while validation RMSE is kept for model-selection context.


In [22]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import random
import shutil
import sys
from datetime import datetime
from pathlib import Path
import time

import numpy as np
import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return candidate
        nested = candidate / "ariel" / "models" / "notebooks"
        if (nested / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the Ariel notebook project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT.parent.parent
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "fair_small_compare" / RUN_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
TRAIN_ROWS = 256
VALIDATION_ROWS = 64
HOLDOUT_ROWS = 64

# Convergence-oriented runtime cap.
# Runtime should bind before max_epochs, while early stopping can still end a run that has clearly converged.
FAIR_RUNTIME_BUDGET_SECONDS = 180.0
CODEX_STAGE1_BUDGET_FRACTION = 0.25
CODEX_STAGE1_BUDGET_SECONDS = FAIR_RUNTIME_BUDGET_SECONDS * CODEX_STAGE1_BUDGET_FRACTION

# Match the main ariel_codex notebook hyperparameters, but allow longer capped runs.
CODEX_QNN_QUBITS = 8
CODEX_QNN_DEPTH = 2
CODEX_DROPOUT = 0.05
CODEX_WEIGHT_DECAY = 1.0e-4
CODEX_GRADIENT_CLIP = 5.0
CODEX_QNN_INIT_SCALE = 0.1
CODEX_USE_AMP = True
CODEX_LOG_EVERY_BATCHES = 20
CODEX_STAGE1_BATCH_SIZE = 256
CODEX_STAGE1_EVAL_BATCH_SIZE = 512
CODEX_STAGE1_MAX_EPOCHS = 120
CODEX_STAGE1_EARLY_STOP = 20
CODEX_STAGE1_SCHEDULER_PATIENCE = 4
CODEX_STAGE1_CLASSICAL_LR = 1.0e-3
CODEX_STAGE2_BATCH_SIZE = 16
CODEX_STAGE2_EVAL_BATCH_SIZE = 32
CODEX_STAGE2_MAX_EPOCHS = 120
CODEX_STAGE2_EARLY_STOP = 20
CODEX_STAGE2_SCHEDULER_PATIENCE = 5
CODEX_STAGE2_CLASSICAL_LR = 5.0e-5
CODEX_STAGE2_QUANTUM_LR = 2.0e-4
CODEX_STAGE2_WARMUP_EPOCHS = 0
CODEX_STAGE2_RAMP_EPOCHS = 12
CODEX_STAGE2_FREEZE_EPOCHS = 6

# FMPE SOTA configuration.
FMPE_MAX_EPOCHS = 200
FMPE_MAX_STEPS = None
FMPE_MAX_RUNTIME_SECONDS = FAIR_RUNTIME_BUDGET_SECONDS
FMPE_PATIENCE = 25
FMPE_BATCH_SIZE = 64
FMPE_EVAL_BATCH_SIZE = 128
FMPE_POSTERIOR_SAMPLES = 16
FMPE_PERIODIC_SAMPLES = 4
FMPE_PERIODIC_RMSE_EVERY_EPOCHS = 5
FMPE_PERIODIC_MAX_ROWS = 32
FMPE_CONTEXT_BATCH_SIZE = 16

DEVICE = torch.device("cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cpu_threads = max(1, min(os.cpu_count() or 1, 8))
torch.set_num_threads(cpu_threads)
try:
    torch.set_num_interop_threads(max(1, min(cpu_threads // 2, 4)))
except RuntimeError:
    pass

config = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "platform": platform.platform(),
    "seed": SEED,
    "train_rows": TRAIN_ROWS,
    "validation_rows": VALIDATION_ROWS,
    "holdout_rows": HOLDOUT_ROWS,
    "codex_qubits": CODEX_QNN_QUBITS,
    "codex_depth": CODEX_QNN_DEPTH,
    "fair_runtime_budget_seconds": FAIR_RUNTIME_BUDGET_SECONDS,
    "codex_stage1_budget_fraction": CODEX_STAGE1_BUDGET_FRACTION,
    "codex_stage1_budget_seconds": CODEX_STAGE1_BUDGET_SECONDS,
    "codex_stage1_epochs": CODEX_STAGE1_MAX_EPOCHS,
    "codex_stage2_epochs": CODEX_STAGE2_MAX_EPOCHS,
    "fmpe_epochs": FMPE_MAX_EPOCHS,
    "fmpe_max_runtime_seconds": FMPE_MAX_RUNTIME_SECONDS,
    "fmpe_max_steps": FMPE_MAX_STEPS,
    "fmpe_periodic_rmse_every_epochs": FMPE_PERIODIC_RMSE_EVERY_EPOCHS,
    "fmpe_periodic_rmse_max_rows": FMPE_PERIODIC_MAX_ROWS,
    "fmpe_posterior_samples": FMPE_POSTERIOR_SAMPLES,
    "fmpe_periodic_posterior_samples": FMPE_PERIODIC_SAMPLES,
}
config


{'project_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks',
 'data_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel',
 'output_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/fair_small_compare/20260312_092858',
 'platform': 'macOS-26.2-arm64-arm-64bit',
 'seed': 42,
 'train_rows': 256,
 'validation_rows': 64,
 'holdout_rows': 64,
 'codex_qubits': 8,
 'codex_depth': 2,
 'fair_runtime_budget_seconds': 180.0,
 'codex_stage1_budget_fraction': 0.25,
 'codex_stage1_budget_seconds': 45.0,
 'codex_stage1_epochs': 120,
 'codex_stage2_epochs': 120,
 'fmpe_epochs': 200,
 'fmpe_max_runtime_seconds': 180.0,
 'fmpe_max_steps': None,
 'fmpe_periodic_rmse_every_epochs': 5,
 'fmpe_periodic_rmse_max_rows': 32,
 'fmpe_posterior_samples': 16,
 'fmpe_periodic_posterior_samples': 4}

In [23]:
required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "h5py": "h5py",
    "PyYAML": "yaml",
    "PyTorch": "torch",
    "scikit-learn": "sklearn",
    "PennyLane": "pennylane",
    "Dingo": "dingo",
}
missing = [label for label, module_name in required_modules.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Missing Python modules:", ", ".join(missing))
    print()
    print("Suggested local bootstrap commands from", PROJECT_ROOT)
    print("python -m pip install --upgrade pip")
    print("python -m pip install torch")
    print("python -m pip install --no-deps dingo-gw==0.8.3")
    print(f"python -m pip install -r {PROJECT_ROOT / 'models' / 'sbi_ariel_crossgen' / 'requirements-mac.txt'}")
    print("python -m pip install pennylane pennylane-lightning")
    raise RuntimeError("Install the missing dependencies, then rerun this cell.")

import h5py
import pandas as pd
import pennylane as qml
import yaml
from IPython.display import display
from sklearn.model_selection import train_test_split

from models.sbi_ariel_adc2023.constants import (
    AUX_FEATURE_COLS as FMPE_AUX_FEATURE_COLS,
    CONTEXT_DIM,
    CONTEXT_FILENAME_TEMPLATE,
    DATASET_TYPE,
    HOLDOUT_SPLIT,
    LOG10_AUX_FEATURE_COLS as FMPE_LOG10_AUX_FEATURE_COLS,
    MANIFEST_FILENAME,
    METADATA_FILENAME_TEMPLATE,
    NORMALIZATION_FILENAME,
    NORMALIZATION_MODE,
    RAW_TARGET_FILENAME_TEMPLATE,
    TARGET_COLS,
    TARGET_FILENAME_TEMPLATE,
    TESTDATA_SPLIT,
    THETA_DIM,
    TRAIN_SPLIT,
    VALIDATION_SPLIT,
    WAVELENGTH_FILENAME,
)
from models.sbi_ariel_adc2023.dataset import load_datasets
from models.sbi_ariel_adc2023.evaluate import run_regression_evaluation
from models.ariel_quantum_regression.constants import (
    AUX_COLUMNS as CODEX_AUX_COLUMNS,
    FIXED_SPECTRAL_CHANNELS,
    HDF5_GROUP_PREFIX,
    LOG10_AUX_COLUMNS as CODEX_LOG10_AUX_COLUMNS,
    MODEL_SPECTRAL_CHANNELS,
    PRESENCE_THRESHOLD_LOG10_VMR,
    RAW_SPECTRAL_CHANNELS,
    SAMPLE_SPECTRAL_CHANNELS,
    TARGET_COLUMNS as CODEX_TARGET_COLUMNS,
    WAVELENGTH_DATASET,
)
from models.ariel_quantum_regression.dataset import (
    ArrayStandardizer,
    PreparedData,
    SpectralStandardizer,
    _expected_manifest,
    _make_inference_split,
    _make_labeled_split,
    _normalize_fixed_channel,
    _normalize_sample_spectra,
    _save_prepared_cache,
    build_stratify_labels,
    transform_aux_features,
)

import importlib
import models.sbi_ariel_adc2023.training as fmpe_training_module
import models.ariel_quantum_regression.training as codex_training_module

fmpe_training_module = importlib.reload(fmpe_training_module)
codex_training_module = importlib.reload(codex_training_module)

train_model = fmpe_training_module.train_model
TrainingConfig = codex_training_module.TrainingConfig
run_training_experiment = codex_training_module.run_training_experiment

QUANTUM_DEVICE = "lightning.gpu" if torch.cuda.is_available() else "lightning.qubit"

print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")
print(f"Device: {DEVICE} | Quantum backend: {QUANTUM_DEVICE}")


PyTorch: 2.10.0
PennyLane: 0.44.1
Device: cpu | Quantum backend: lightning.qubit


In [24]:
def _drop_unnamed_columns(frame: pd.DataFrame) -> pd.DataFrame:
    unnamed = [column for column in frame.columns if column.startswith("Unnamed:")]
    if unnamed:
        frame = frame.drop(columns=unnamed)
    return frame


def _load_spectra(hdf5_path: Path, planet_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    spectra = np.empty((len(planet_ids), 52, len(RAW_SPECTRAL_CHANNELS)), dtype=np.float32)
    wavelength_um = None
    with h5py.File(hdf5_path, "r") as handle:
        for row_index, planet_id in enumerate(planet_ids.tolist()):
            group = handle[f"{HDF5_GROUP_PREFIX}{planet_id}"]
            row_wavelength = np.asarray(group[WAVELENGTH_DATASET][:], dtype=np.float32)
            if wavelength_um is None:
                wavelength_um = row_wavelength
            elif not np.allclose(row_wavelength, wavelength_um, atol=1.0e-8):
                raise AssertionError(f"Wavelength mismatch for {planet_id}")
            for channel_index, channel_name in enumerate(RAW_SPECTRAL_CHANNELS):
                spectra[row_index, :, channel_index] = np.asarray(group[channel_name][:], dtype=np.float32)
    if wavelength_um is None:
        raise RuntimeError(f"No spectra found in {hdf5_path}")
    return spectra, wavelength_um


train_aux_df = _drop_unnamed_columns(pd.read_csv(DATA_ROOT / "TrainingData" / "AuxillaryTable.csv"))
train_targets_df = _drop_unnamed_columns(
    pd.read_csv(DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv")
)
labels = train_aux_df.merge(train_targets_df[["planet_ID", *TARGET_COLS]], on="planet_ID", how="inner", validate="one_to_one")
labels = labels.reset_index(drop=True)

train_spectra_raw, wavelength_um = _load_spectra(
    DATA_ROOT / "TrainingData" / "SpectralData.hdf5",
    labels["planet_ID"].to_numpy(dtype="U32"),
)
test_aux_df = _drop_unnamed_columns(pd.read_csv(DATA_ROOT / "TestData" / "AuxillaryTable.csv"))
test_spectra_raw, test_wavelength_um = _load_spectra(
    DATA_ROOT / "TestData" / "SpectralData.hdf5",
    test_aux_df["planet_ID"].to_numpy(dtype="U32"),
)
if not np.allclose(wavelength_um, test_wavelength_um, atol=1.0e-8):
    raise AssertionError("Training and test wavelength grids do not match.")

targets_raw = labels[TARGET_COLS].to_numpy(dtype=np.float32, copy=True)
print(f"Training rows: {len(labels):,}")
print(f"Challenge test rows: {len(test_aux_df):,}")
print(f"Wavelength bins: {len(wavelength_um)}")


Training rows: 41,423
Challenge test rows: 685
Wavelength bins: 52


In [25]:
all_indices = np.arange(len(labels), dtype=np.int64)
stratify_all, stratify_mode_all = build_stratify_labels(targets_raw)
train_pool, temp_pool = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
    stratify=stratify_all if stratify_all is not None else None,
)
temp_targets_raw = targets_raw[temp_pool]
stratify_temp, stratify_mode_temp = build_stratify_labels(temp_targets_raw)
temp_positions = np.arange(len(temp_pool), dtype=np.int64)
validation_positions, holdout_positions = train_test_split(
    temp_positions,
    test_size=0.5,
    random_state=SEED + 1,
    shuffle=True,
    stratify=stratify_temp if stratify_temp is not None else None,
)

validation_pool = temp_pool[validation_positions]
holdout_pool = temp_pool[holdout_positions]
rng = np.random.default_rng(SEED)


def take_subset(indices: np.ndarray, limit: int) -> np.ndarray:
    if limit >= len(indices):
        return np.sort(indices.astype(np.int64))
    chosen = rng.choice(indices, size=int(limit), replace=False)
    return np.sort(chosen.astype(np.int64))


train_idx = take_subset(train_pool, TRAIN_ROWS)
validation_idx = take_subset(validation_pool, VALIDATION_ROWS)
holdout_idx = take_subset(holdout_pool, HOLDOUT_ROWS)

split_frame = pd.DataFrame(
    {
        "split": ([TRAIN_SPLIT] * len(train_idx)) + ([VALIDATION_SPLIT] * len(validation_idx)) + ([HOLDOUT_SPLIT] * len(holdout_idx)),
        "planet_ID": np.concatenate(
            [
                labels.iloc[train_idx]["planet_ID"].to_numpy(dtype="U32"),
                labels.iloc[validation_idx]["planet_ID"].to_numpy(dtype="U32"),
                labels.iloc[holdout_idx]["planet_ID"].to_numpy(dtype="U32"),
            ]
        ),
        "source_row_index": np.concatenate([train_idx, validation_idx, holdout_idx]),
    }
)
SHARED_SPLIT_PATH = OUTPUT_ROOT / "shared_split.csv"
split_frame.to_csv(SHARED_SPLIT_PATH, index=False)

shared_split_manifest = {
    "seed": SEED,
    "stratify_mode_train_split": stratify_mode_all,
    "stratify_mode_validation_split": stratify_mode_temp,
    "train_rows": int(len(train_idx)),
    "validation_rows": int(len(validation_idx)),
    "holdout_rows": int(len(holdout_idx)),
    "shared_split_file": str(SHARED_SPLIT_PATH),
}
(OUTPUT_ROOT / "shared_split_manifest.json").write_text(json.dumps(shared_split_manifest, indent=2, sort_keys=True) + "\n")
split_frame.groupby("split").size().to_frame("rows")


,rows
split,
holdout,64
train,256
validation,64


In [26]:
def safe_scale(values: np.ndarray) -> np.ndarray:
    scale = values.std(axis=0, ddof=0).astype(np.float32)
    return np.where(scale == 0.0, 1.0, scale).astype(np.float32)


def normalize_spectrum(values: np.ndarray) -> np.ndarray:
    sample_mean = values.mean(axis=1, keepdims=True)
    sample_mean = np.clip(sample_mean, 1.0e-12, None)
    return (values / sample_mean).astype(np.float32)


def log_noise(values: np.ndarray) -> np.ndarray:
    return np.log10(np.clip(values.astype(np.float32), 1.0e-12, None)).astype(np.float32)


def compute_zscore(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = values.mean(axis=0).astype(np.float32)
    scale = safe_scale(values)
    return mean, scale


def apply_zscore(values: np.ndarray, mean: np.ndarray, scale: np.ndarray) -> np.ndarray:
    return ((values - mean) / scale).astype(np.float32)


def metadata_frame(indices: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "planet_ID": labels.iloc[indices]["planet_ID"].to_numpy(dtype="U32"),
            "source_row_index": indices.astype(np.int64),
        }
    )


CODEX_PREPARED_CACHE_DIR = OUTPUT_ROOT / "codex_prepared_cache"
if CODEX_PREPARED_CACHE_DIR.exists():
    shutil.rmtree(CODEX_PREPARED_CACHE_DIR)
CODEX_PREPARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

codex_aux_raw = transform_aux_features(labels)
codex_test_aux_raw = transform_aux_features(test_aux_df)
codex_targets_raw = labels[CODEX_TARGET_COLUMNS].to_numpy(dtype=np.float32, copy=True)
sample_channel_indices = [RAW_SPECTRAL_CHANNELS.index(name) for name in SAMPLE_SPECTRAL_CHANNELS]
width_channel_index = RAW_SPECTRAL_CHANNELS.index("instrument_width")
codex_train_sample_spectra = np.transpose(train_spectra_raw[:, :, sample_channel_indices], (0, 2, 1)).astype(np.float32)
codex_test_sample_spectra = np.transpose(test_spectra_raw[:, :, sample_channel_indices], (0, 2, 1)).astype(np.float32)
codex_train_sample_spectra = _normalize_sample_spectra(codex_train_sample_spectra)
codex_test_sample_spectra = _normalize_sample_spectra(codex_test_sample_spectra)
fixed_channels = np.stack(
    [
        _normalize_fixed_channel(train_spectra_raw[0, :, width_channel_index]),
        _normalize_fixed_channel(wavelength_um),
    ],
    axis=0,
).astype(np.float32)
codex_aux_scaler = ArrayStandardizer.fit(codex_aux_raw[train_idx])
codex_target_scaler = ArrayStandardizer.fit(codex_targets_raw[train_idx])
codex_spectral_scaler = SpectralStandardizer.fit(codex_train_sample_spectra[train_idx], fixed_channels=fixed_channels)


def make_codex_labeled_split(indices: np.ndarray):
    aux_scaled = codex_aux_scaler.transform(codex_aux_raw[indices])
    spectra_scaled = codex_spectral_scaler.transform(codex_train_sample_spectra[indices])
    raw_targets = codex_targets_raw[indices]
    targets_scaled = codex_target_scaler.transform(raw_targets)
    return _make_labeled_split(
        planet_ids=labels.iloc[indices]["planet_ID"].to_numpy(dtype="U32"),
        aux_values=aux_scaled,
        spectra_values=spectra_scaled,
        targets_scaled=targets_scaled,
        raw_targets=raw_targets,
    )


codex_test_split = _make_inference_split(
    planet_ids=test_aux_df["planet_ID"].to_numpy(dtype="U32"),
    aux_values=codex_aux_scaler.transform(codex_test_aux_raw),
    spectra_values=codex_spectral_scaler.transform(codex_test_sample_spectra),
)
codex_split_manifest = {
    "rows_total": int(len(labels)),
    "testdata_rows_total": int(len(test_aux_df)),
    "train_rows": int(len(train_idx)),
    "val_rows": int(len(validation_idx)),
    "holdout_rows": int(len(holdout_idx)),
    "testdata_rows": int(len(test_aux_df)),
    "wavelength_bins": int(len(wavelength_um)),
    "wavelength_min_um": float(wavelength_um.min()),
    "wavelength_max_um": float(wavelength_um.max()),
    "raw_spectrum_shape": [52, len(RAW_SPECTRAL_CHANNELS)],
    "model_spectrum_shape": [len(MODEL_SPECTRAL_CHANNELS), 52],
    "aux_columns": CODEX_AUX_COLUMNS,
    "log10_aux_columns": CODEX_LOG10_AUX_COLUMNS,
    "target_columns": CODEX_TARGET_COLUMNS,
    "raw_spectral_channels": RAW_SPECTRAL_CHANNELS,
    "sample_spectral_channels": SAMPLE_SPECTRAL_CHANNELS,
    "fixed_spectral_channels": FIXED_SPECTRAL_CHANNELS,
    "model_spectral_channels": MODEL_SPECTRAL_CHANNELS,
    "sample_spectral_normalization": {
        "mode": "divide_by_sample_mean",
        "reference_channel": SAMPLE_SPECTRAL_CHANNELS[0],
        "applied_channels": SAMPLE_SPECTRAL_CHANNELS,
    },
    "presence_threshold_log10_vmr": PRESENCE_THRESHOLD_LOG10_VMR,
    "split_seed": int(SEED),
    "split_fractions": {"train": 0.8, "val": 0.1, "holdout": 0.1},
    "primary_stratify_mode": stratify_mode_all,
    "secondary_stratify_mode": stratify_mode_temp,
    "shared_split_file": str(SHARED_SPLIT_PATH),
}
codex_expected_manifest = _expected_manifest(
    DATA_ROOT,
    SEED,
    int(len(train_idx)),
    int(len(validation_idx)),
    int(len(holdout_idx)),
    None,
)
codex_prepared_manifest = dict(codex_expected_manifest)
codex_prepared_manifest["cache_dir"] = str(CODEX_PREPARED_CACHE_DIR)
codex_prepared_manifest["cache_hit"] = False
codex_prepared = PreparedData(
    train=make_codex_labeled_split(train_idx),
    val=make_codex_labeled_split(validation_idx),
    holdout=make_codex_labeled_split(holdout_idx),
    testdata=codex_test_split,
    aux_scaler=codex_aux_scaler,
    target_scaler=codex_target_scaler,
    spectral_scaler=codex_spectral_scaler,
    wavelength_um=wavelength_um.astype(np.float32),
    split_manifest=codex_split_manifest,
    prepared_manifest=codex_prepared_manifest,
)
_save_prepared_cache(CODEX_PREPARED_CACHE_DIR, codex_expected_manifest, codex_prepared)

FMPE_PREPARED_DIR = OUTPUT_ROOT / "fmpe_prepared_dataset"
if FMPE_PREPARED_DIR.exists():
    shutil.rmtree(FMPE_PREPARED_DIR)
FMPE_PREPARED_DIR.mkdir(parents=True, exist_ok=True)
fmpe_labeled_spectrum = normalize_spectrum(train_spectra_raw[:, :, RAW_SPECTRAL_CHANNELS.index("instrument_spectrum")])
fmpe_test_spectrum = normalize_spectrum(test_spectra_raw[:, :, RAW_SPECTRAL_CHANNELS.index("instrument_spectrum")])
fmpe_labeled_noise = log_noise(train_spectra_raw[:, :, RAW_SPECTRAL_CHANNELS.index("instrument_noise")])
fmpe_test_noise = log_noise(test_spectra_raw[:, :, RAW_SPECTRAL_CHANNELS.index("instrument_noise")])
fmpe_labeled_aux = labels[FMPE_AUX_FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
fmpe_test_aux = test_aux_df[FMPE_AUX_FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
for col_index, col_name in enumerate(FMPE_AUX_FEATURE_COLS):
    if col_name in FMPE_LOG10_AUX_FEATURE_COLS:
        fmpe_labeled_aux[:, col_index] = np.log10(np.clip(fmpe_labeled_aux[:, col_index], 1.0e-12, None))
        fmpe_test_aux[:, col_index] = np.log10(np.clip(fmpe_test_aux[:, col_index], 1.0e-12, None))
fmpe_targets_raw = labels[TARGET_COLS].to_numpy(dtype=np.float32, copy=True)
spectrum_mean, spectrum_scale = compute_zscore(fmpe_labeled_spectrum[train_idx])
noise_mean, noise_scale = compute_zscore(fmpe_labeled_noise[train_idx])
aux_mean, aux_scale = compute_zscore(fmpe_labeled_aux[train_idx])
target_mean, target_scale = compute_zscore(fmpe_targets_raw[train_idx])


def fmpe_context(indices: np.ndarray) -> np.ndarray:
    return np.concatenate(
        [
            apply_zscore(fmpe_labeled_spectrum[indices], spectrum_mean, spectrum_scale),
            apply_zscore(fmpe_labeled_noise[indices], noise_mean, noise_scale),
            apply_zscore(fmpe_labeled_aux[indices], aux_mean, aux_scale),
        ],
        axis=1,
    ).astype(np.float32)


def write_fmpe_labeled_split(split_name: str, indices: np.ndarray) -> None:
    np.save(FMPE_PREPARED_DIR / CONTEXT_FILENAME_TEMPLATE.format(split_name=split_name), fmpe_context(indices))
    np.save(
        FMPE_PREPARED_DIR / TARGET_FILENAME_TEMPLATE.format(split_name=split_name),
        apply_zscore(fmpe_targets_raw[indices], target_mean, target_scale),
    )
    np.save(
        FMPE_PREPARED_DIR / RAW_TARGET_FILENAME_TEMPLATE.format(split_name=split_name),
        fmpe_targets_raw[indices].astype(np.float32),
    )
    metadata_frame(indices).to_csv(
        FMPE_PREPARED_DIR / METADATA_FILENAME_TEMPLATE.format(split_name=split_name),
        index=False,
    )


write_fmpe_labeled_split(TRAIN_SPLIT, train_idx)
write_fmpe_labeled_split(VALIDATION_SPLIT, validation_idx)
write_fmpe_labeled_split(HOLDOUT_SPLIT, holdout_idx)
test_context = np.concatenate(
    [
        apply_zscore(fmpe_test_spectrum, spectrum_mean, spectrum_scale),
        apply_zscore(fmpe_test_noise, noise_mean, noise_scale),
        apply_zscore(fmpe_test_aux, aux_mean, aux_scale),
    ],
    axis=1,
).astype(np.float32)
np.save(FMPE_PREPARED_DIR / CONTEXT_FILENAME_TEMPLATE.format(split_name=TESTDATA_SPLIT), test_context)
pd.DataFrame(
    {
        "planet_ID": test_aux_df["planet_ID"].to_numpy(dtype="U32"),
        "source_row_index": np.arange(len(test_aux_df), dtype=np.int64),
    }
).to_csv(FMPE_PREPARED_DIR / METADATA_FILENAME_TEMPLATE.format(split_name=TESTDATA_SPLIT), index=False)
np.savez(
    FMPE_PREPARED_DIR / NORMALIZATION_FILENAME,
    spectrum_mean=spectrum_mean,
    spectrum_scale=spectrum_scale,
    noise_mean=noise_mean,
    noise_scale=noise_scale,
    aux_mean=aux_mean,
    aux_scale=aux_scale,
    target_mean=target_mean,
    target_scale=target_scale,
)
np.save(FMPE_PREPARED_DIR / WAVELENGTH_FILENAME, wavelength_um.astype(np.float32))
fmpe_manifest = {
    "data_root": str(DATA_ROOT),
    "prepared_dir": str(FMPE_PREPARED_DIR),
    "dataset_type": DATASET_TYPE,
    "normalization_mode": NORMALIZATION_MODE,
    "seed": SEED,
    "split_fractions": {"train": None, "validation": None, "holdout": None},
    "split_counts": {
        TRAIN_SPLIT: int(len(train_idx)),
        VALIDATION_SPLIT: int(len(validation_idx)),
        HOLDOUT_SPLIT: int(len(holdout_idx)),
        TESTDATA_SPLIT: int(len(test_aux_df)),
    },
    "context_dim": CONTEXT_DIM,
    "theta_dim": THETA_DIM,
    "target_columns": TARGET_COLS,
    "aux_feature_cols": FMPE_AUX_FEATURE_COLS,
    "log10_aux_feature_cols": FMPE_LOG10_AUX_FEATURE_COLS,
    "wavelength_bins": int(len(wavelength_um)),
    "wavelength_min_um": float(wavelength_um.min()),
    "wavelength_max_um": float(wavelength_um.max()),
    "feature_order": [
        "normalized_instrument_spectrum[52]",
        "log10_instrument_noise[52]",
        *FMPE_AUX_FEATURE_COLS,
    ],
    "shared_split_file": str(SHARED_SPLIT_PATH),
}
(FMPE_PREPARED_DIR / MANIFEST_FILENAME).write_text(json.dumps(fmpe_manifest, indent=2, sort_keys=True) + "\n")

print(f"Codex prepared cache: {CODEX_PREPARED_CACHE_DIR}")
print(f"FMPE prepared dataset: {FMPE_PREPARED_DIR}")


Codex prepared cache: /Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/fair_small_compare/20260312_092858/codex_prepared_cache
FMPE prepared dataset: /Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/fair_small_compare/20260312_092858/fmpe_prepared_dataset


In [27]:
CODEX_OUTPUT_ROOT = OUTPUT_ROOT / "codex"
if CODEX_OUTPUT_ROOT.exists():
    shutil.rmtree(CODEX_OUTPUT_ROOT)
CODEX_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CODEX_STAGE1_OUTPUT_DIR = CODEX_OUTPUT_ROOT / "stage1_classical"
CODEX_STAGE2_OUTPUT_DIR = CODEX_OUTPUT_ROOT / "stage2_hybrid"


def make_codex_stage1_config(output_dir: Path, runtime_seconds: float) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(CODEX_PREPARED_CACHE_DIR),
        seed=SEED,
        batch_size=CODEX_STAGE1_BATCH_SIZE,
        eval_batch_size=CODEX_STAGE1_EVAL_BATCH_SIZE,
        max_epochs=CODEX_STAGE1_MAX_EPOCHS,
        max_runtime_seconds=runtime_seconds,
        early_stop_patience=CODEX_STAGE1_EARLY_STOP,
        scheduler_patience=CODEX_STAGE1_SCHEDULER_PATIENCE,
        scheduler_factor=0.5,
        classical_lr=CODEX_STAGE1_CLASSICAL_LR,
        weight_decay=CODEX_WEIGHT_DECAY,
        gradient_clip_norm=CODEX_GRADIENT_CLIP,
        dropout=CODEX_DROPOUT,
        loss_name="mse",
        qnn_qubits=CODEX_QNN_QUBITS,
        qnn_depth=CODEX_QNN_DEPTH,
        qnn_init_scale=CODEX_QNN_INIT_SCALE,
        quantum_device=QUANTUM_DEVICE,
        quantum_use_async=False,
        classical_only=True,
        use_amp=CODEX_USE_AMP,
        log_every_batches=CODEX_LOG_EVERY_BATCHES,
        train_limit=len(train_idx),
        val_limit=len(validation_idx),
        holdout_limit=len(holdout_idx),
    )


def make_codex_stage2_config(output_dir: Path, checkpoint_path: Path, runtime_seconds: float) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(CODEX_PREPARED_CACHE_DIR),
        init_checkpoint_path=str(checkpoint_path),
        seed=SEED,
        batch_size=CODEX_STAGE2_BATCH_SIZE,
        eval_batch_size=CODEX_STAGE2_EVAL_BATCH_SIZE,
        max_epochs=CODEX_STAGE2_MAX_EPOCHS,
        max_runtime_seconds=runtime_seconds,
        early_stop_patience=CODEX_STAGE2_EARLY_STOP,
        scheduler_patience=CODEX_STAGE2_SCHEDULER_PATIENCE,
        scheduler_factor=0.5,
        classical_lr=CODEX_STAGE2_CLASSICAL_LR,
        quantum_lr=CODEX_STAGE2_QUANTUM_LR,
        weight_decay=CODEX_WEIGHT_DECAY,
        gradient_clip_norm=CODEX_GRADIENT_CLIP,
        dropout=CODEX_DROPOUT,
        loss_name="mse",
        qnn_qubits=CODEX_QNN_QUBITS,
        qnn_depth=CODEX_QNN_DEPTH,
        qnn_init_scale=CODEX_QNN_INIT_SCALE,
        quantum_device=QUANTUM_DEVICE,
        quantum_use_async=False,
        quantum_warmup_epochs=CODEX_STAGE2_WARMUP_EPOCHS,
        quantum_ramp_epochs=CODEX_STAGE2_RAMP_EPOCHS,
        quantum_backbone_freeze_epochs=CODEX_STAGE2_FREEZE_EPOCHS,
        classical_only=False,
        use_amp=CODEX_USE_AMP,
        log_every_batches=CODEX_LOG_EVERY_BATCHES,
        train_limit=len(train_idx),
        val_limit=len(validation_idx),
        holdout_limit=len(holdout_idx),
    )


codex_training_start = time.perf_counter()
codex_stage1_cfg = make_codex_stage1_config(CODEX_STAGE1_OUTPUT_DIR, CODEX_STAGE1_BUDGET_SECONDS)
print(json.dumps(codex_stage1_cfg.to_json_dict(), indent=2))
codex_stage1_result = run_training_experiment(codex_stage1_cfg)
print(json.dumps(codex_stage1_result["summary"], indent=2))

codex_stage1_checkpoint = (CODEX_STAGE1_OUTPUT_DIR / "best_model.pt").resolve()
if not codex_stage1_checkpoint.exists():
    raise FileNotFoundError(f"Missing Codex stage-1 checkpoint: {codex_stage1_checkpoint}")

codex_stage2_budget_seconds = max(1.0, FAIR_RUNTIME_BUDGET_SECONDS - (time.perf_counter() - codex_training_start))
codex_stage2_cfg = make_codex_stage2_config(CODEX_STAGE2_OUTPUT_DIR, codex_stage1_checkpoint, codex_stage2_budget_seconds)
print(json.dumps(codex_stage2_cfg.to_json_dict(), indent=2))
codex_stage2_result = run_training_experiment(codex_stage2_cfg)
print(json.dumps(codex_stage2_result["summary"], indent=2))

codex_result = {
    "model": "codex_two_stage_8q",
    "best_epoch": int(codex_stage2_result["summary"]["best_epoch"]),
    "best_val_rmse_mean": float(codex_stage2_result["summary"]["best_val_rmse_mean"]),
    "validation_rmse_mean": float(codex_stage2_result["summary"]["validation_rmse_mean"]),
    "holdout_rmse_mean": float(codex_stage2_result["summary"]["holdout_rmse_mean"]),
    "validation_rmse": {
        name: float(value)
        for name, value in zip(CODEX_TARGET_COLUMNS, codex_stage2_result["validation_metrics"]["rmse_orig"])
    },
    "holdout_rmse": {
        name: float(value)
        for name, value in zip(CODEX_TARGET_COLUMNS, codex_stage2_result["holdout_metrics"]["rmse_orig"])
    },
    "hyperparameters": {
        "qnn_qubits": CODEX_QNN_QUBITS,
        "qnn_depth": CODEX_QNN_DEPTH,
        "use_amp": CODEX_USE_AMP,
        "log_every_batches": CODEX_LOG_EVERY_BATCHES,
        "total_runtime_budget_seconds": FAIR_RUNTIME_BUDGET_SECONDS,
        "stage1_runtime_budget_seconds": CODEX_STAGE1_BUDGET_SECONDS,
        "stage2_runtime_budget_seconds": codex_stage2_budget_seconds,
        "stage1_batch_size": CODEX_STAGE1_BATCH_SIZE,
        "stage1_epochs": CODEX_STAGE1_MAX_EPOCHS,
        "stage1_classical_lr": CODEX_STAGE1_CLASSICAL_LR,
        "stage2_batch_size": CODEX_STAGE2_BATCH_SIZE,
        "stage2_epochs": CODEX_STAGE2_MAX_EPOCHS,
        "stage2_classical_lr": CODEX_STAGE2_CLASSICAL_LR,
        "stage2_quantum_lr": CODEX_STAGE2_QUANTUM_LR,
        "stage2_quantum_ramp_epochs": CODEX_STAGE2_RAMP_EPOCHS,
        "stage2_quantum_backbone_freeze_epochs": CODEX_STAGE2_FREEZE_EPOCHS,
    },
    "stage1_summary": codex_stage1_result["summary"],
    "stage2_summary": codex_stage2_result["summary"],
}
(CODEX_OUTPUT_ROOT / "summary.json").write_text(json.dumps(codex_result, indent=2, sort_keys=True) + "\n")
codex_result


{
  "project_root": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks",
  "data_root": "/Users/jkw/Documents/uni/axion/hack4sages/ariel",
  "output_dir": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/fair_small_compare/20260312_092858/codex/stage1_classical",
  "prepared_cache_dir": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/fair_small_compare/20260312_092858/codex_prepared_cache",
  "init_checkpoint_path": null,
  "seed": 42,
  "batch_size": 256,
  "eval_batch_size": 512,
  "max_epochs": 120,
  "max_runtime_seconds": 45.0,
  "early_stop_patience": 20,
  "scheduler_patience": 4,
  "scheduler_factor": 0.5,
  "classical_lr": 0.001,
  "quantum_lr": 0.0008,
  "weight_decay": 0.0001,
  "gradient_clip_norm": 5.0,
  "dropout": 0.05,
  "loss_name": "mse",
  "qnn_qubits": 8,
  "qnn_depth": 2,
  "qnn_init_scale": 0.1,
  "quantum_device": "lightning.qubit",
  "quantum_use_async": false,
  "classical_only": true,
  "quantum

{'model': 'codex_two_stage_8q',
 'best_epoch': 1,
 'best_val_rmse_mean': 0.9214053153991699,
 'validation_rmse_mean': 0.9267433285713196,
 'holdout_rmse_mean': 0.8358033299446106,
 'validation_rmse': {'log_H2O': 1.137733817100525,
  'log_CO2': 0.9354819655418396,
  'log_CO': 0.8521819114685059,
  'log_CH4': 0.7129623889923096,
  'log_NH3': 0.9953564405441284},
 'holdout_rmse': {'log_H2O': 1.032458782196045,
  'log_CO2': 0.8739594221115112,
  'log_CO': 0.7487684488296509,
  'log_CH4': 0.5904542207717896,
  'log_NH3': 0.9333756566047668},
 'hyperparameters': {'qnn_qubits': 8,
  'qnn_depth': 2,
  'use_amp': True,
  'log_every_batches': 20,
  'total_runtime_budget_seconds': 180.0,
  'stage1_runtime_budget_seconds': 45.0,
  'stage2_runtime_budget_seconds': 169.87682729100925,
  'stage1_batch_size': 256,
  'stage1_epochs': 120,
  'stage1_classical_lr': 0.001,
  'stage2_batch_size': 16,
  'stage2_epochs': 120,
  'stage2_classical_lr': 5e-05,
  'stage2_quantum_lr': 0.0002,
  'stage2_quantum_ra

In [ ]:
FMPE_OUTPUT_DIR = OUTPUT_ROOT / "fmpe"
if FMPE_OUTPUT_DIR.exists():
    shutil.rmtree(FMPE_OUTPUT_DIR)
FMPE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fmpe_settings = yaml.safe_load((PROJECT_ROOT / "models" / "sbi_ariel_adc2023" / "settings" / "adc2023_rtx4090.yaml").read_text())
fmpe_settings["dataset"]["path"] = str(FMPE_PREPARED_DIR)
fmpe_settings["training"]["device"] = "cpu"
fmpe_settings["training"]["batch_size"] = FMPE_BATCH_SIZE
fmpe_settings["training"]["eval_batch_size"] = FMPE_EVAL_BATCH_SIZE
fmpe_settings["training"]["epochs"] = FMPE_MAX_EPOCHS
fmpe_settings["training"]["patience"] = FMPE_PATIENCE
fmpe_settings["training"]["checkpoint_every_batches"] = 5
fmpe_settings["training"]["num_workers"] = 0
fmpe_settings["training"]["pin_memory"] = False
fmpe_settings["training"]["persistent_workers"] = False
fmpe_settings["training"]["prefetch_factor"] = None
fmpe_settings["training"]["max_steps"] = FMPE_MAX_STEPS
fmpe_settings["training"]["max_runtime_seconds"] = FMPE_MAX_RUNTIME_SECONDS
fmpe_settings["evaluation"]["device"] = "cpu"
fmpe_settings["evaluation"]["run_after_training"] = False
fmpe_settings["evaluation"]["posterior_samples"] = FMPE_POSTERIOR_SAMPLES
fmpe_settings["evaluation"]["context_batch_size"] = FMPE_CONTEXT_BATCH_SIZE
fmpe_settings["evaluation"]["progress_every_batches"] = 0
fmpe_settings["evaluation"]["periodic_rmse_every_epochs"] = FMPE_PERIODIC_RMSE_EVERY_EPOCHS
fmpe_settings["evaluation"]["periodic_posterior_samples"] = FMPE_PERIODIC_SAMPLES
fmpe_settings["evaluation"]["periodic_context_batch_size"] = FMPE_CONTEXT_BATCH_SIZE
fmpe_settings["evaluation"]["periodic_rmse_max_rows"] = FMPE_PERIODIC_MAX_ROWS
fmpe_settings["logging"]["use_wandb"] = False
fmpe_settings["task"]["name"] = f"adc2023_fair_small_compare_{RUN_TAG}"

fmpe_training_summary = train_model(
    settings=fmpe_settings,
    run_dir=FMPE_OUTPUT_DIR,
    prepared_data_override=FMPE_PREPARED_DIR,
    resume_mode="never",
)
print(json.dumps(fmpe_training_summary, indent=2, sort_keys=True))

fmpe_eval_outputs = run_regression_evaluation(
    settings=fmpe_settings,
    run_dir=FMPE_OUTPUT_DIR,
    prepared_data_override=FMPE_PREPARED_DIR,
)
print(json.dumps(fmpe_eval_outputs, indent=2, sort_keys=True))

validation_metrics = json.loads((FMPE_OUTPUT_DIR / "validation_regression_metrics.json").read_text())
holdout_metrics = json.loads((FMPE_OUTPUT_DIR / "holdout_regression_metrics.json").read_text())
posterior_selection = json.loads((FMPE_OUTPUT_DIR / "posterior_selection.json").read_text())
point_estimate_mode = posterior_selection["selected_point_estimate"]
fmpe_result = {
    "model": "sota_fmpe",
    "selected_point_estimate": point_estimate_mode,
    "best_epoch": int(fmpe_training_summary["best_epoch"]),
    "best_val_loss": float(fmpe_training_summary["best_val_loss"]),
    "validation_rmse_mean": float(validation_metrics[point_estimate_mode]["rmse_mean"]),
    "holdout_rmse_mean": float(holdout_metrics[point_estimate_mode]["rmse_mean"]),
    "validation_rmse": {name: float(value) for name, value in validation_metrics[point_estimate_mode]["rmse"].items()},
    "holdout_rmse": {name: float(value) for name, value in holdout_metrics[point_estimate_mode]["rmse"].items()},
    "hyperparameters": {
        "epochs": FMPE_MAX_EPOCHS,
        "max_steps": FMPE_MAX_STEPS,
        "max_runtime_seconds": FMPE_MAX_RUNTIME_SECONDS,
        "patience": FMPE_PATIENCE,
        "batch_size": FMPE_BATCH_SIZE,
        "eval_batch_size": FMPE_EVAL_BATCH_SIZE,
        "posterior_samples": FMPE_POSTERIOR_SAMPLES,
        "periodic_posterior_samples": FMPE_PERIODIC_SAMPLES,
        "periodic_rmse_every_epochs": FMPE_PERIODIC_RMSE_EVERY_EPOCHS,
        "periodic_rmse_max_rows": FMPE_PERIODIC_MAX_ROWS,
    },
    "training_summary": fmpe_training_summary,
    "evaluation_outputs": fmpe_eval_outputs,
}
(FMPE_OUTPUT_DIR / "summary.json").write_text(json.dumps(fmpe_result, indent=2, sort_keys=True) + "\n")
fmpe_result


Putting posterior model to device cpu.
Max runtime seconds: 180.0
Start training epoch 1 on cpu.
Epoch 1 complete | train_loss=1.914372 val_loss=1.857651 lr=4.999452e-04 best_val_loss=inf best_rmse=n/a
Start training epoch 2 on cpu.
Epoch 2 complete | train_loss=1.923650 val_loss=1.730284 lr=4.997807e-04 best_val_loss=1.857651 best_rmse=n/a
Start training epoch 3 on cpu.
Epoch 3 complete | train_loss=1.694967 val_loss=1.657778 lr=4.995067e-04 best_val_loss=1.730284 best_rmse=n/a
Start training epoch 4 on cpu.
Epoch 4 complete | train_loss=1.646199 val_loss=1.872782 lr=4.991232e-04 best_val_loss=1.657778 best_rmse=n/a
Start training epoch 5 on cpu.
Epoch 5 complete | train_loss=1.537422 val_loss=1.684791 lr=4.986305e-04 best_val_loss=1.657778 best_rmse=n/a
Epoch 5 RMSE monitor | device=cpu rows=32/64 posterior_samples=4
Epoch 5 RMSE | mean=1.363036 median=1.376942 rows=32 samples=4 device=cpu
Start training epoch 6 on cpu.
Epoch 6 complete | train_loss=1.548868 val_loss=1.678200 lr=4.98

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "model": codex_result["model"],
            "validation_rmse_mean": codex_result["validation_rmse_mean"],
            "holdout_rmse_mean": codex_result["holdout_rmse_mean"],
            "notes": (
                f"two_stage, qubits={CODEX_QNN_QUBITS}, runtime_budget_s={FAIR_RUNTIME_BUDGET_SECONDS:.1f}, "
                f"stage1_status={codex_stage1_result['summary']['status']}, stage2_status={codex_stage2_result['summary']['status']}"
            ),
        },
        {
            "model": fmpe_result["model"],
            "validation_rmse_mean": fmpe_result["validation_rmse_mean"],
            "holdout_rmse_mean": fmpe_result["holdout_rmse_mean"],
            "notes": (
                f"point_estimate={fmpe_result['selected_point_estimate']}, runtime_budget_s={FMPE_MAX_RUNTIME_SECONDS:.1f}, "
                f"status={fmpe_training_summary['status']}"
            ),
        },
    ]
).sort_values("holdout_rmse_mean", ascending=True, ignore_index=True)
winner = comparison_df.iloc[0]
comparison_summary = {
    "winner": winner["model"],
    "comparison": comparison_df.to_dict(orient="records"),
    "shared_split_file": str(SHARED_SPLIT_PATH),
    "shared_split_manifest": str(OUTPUT_ROOT / "shared_split_manifest.json"),
    "codex_prepared_cache_dir": str(CODEX_PREPARED_CACHE_DIR),
    "codex_output_root": str(CODEX_OUTPUT_ROOT),
    "fmpe_prepared_dir": str(FMPE_PREPARED_DIR),
    "fmpe_output_dir": str(FMPE_OUTPUT_DIR),
}
(OUTPUT_ROOT / "comparison_summary.json").write_text(json.dumps(comparison_summary, indent=2, sort_keys=True) + "\n")

display(comparison_df)
print(f"Winner on holdout mean RMSE: {winner['model']}")
print(f"Comparison summary: {OUTPUT_ROOT / 'comparison_summary.json'}")
